   
# Databricks Vector Search(RAG)를 활용한 PDF 문서 기반 지식 베이스 구축

앞서 살펴본 것처럼, 우리의 에이전트는 WIFI 라우터 오류 코드와 같은 구체적이고 기술적인 질문에 대해서는 제대로 작동하지 않았습니다.

이는 에이전트가 우리의 내부 시스템과 제품에 대한 지식을 가지고 있지 않기 때문입니다.

다행히도, 이러한 모든 정보는 PDF 형태로 우리에게 제공되어 있습니다. 이 PDF 파일들은 볼륨에 저장되어 있습니다.

우리는 이 파일들을 파싱하여 Vector Search에 저장한 다음, 에이전트에 리트리버를 추가하여 성능을 개선할 것입니다!


<div style="background-color: #d4e7ff; padding: 10px; border-radius: 15px;">
<strong>참고:</strong> 곧 Databricks Agents를 활용하여 몇 번의 클릭만으로 Knowledge Base를 추가하는 방법을 보여드리겠습니다!
</div>

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-science&org_id=601859075515029&notebook=%2F03-knowledge-base-rag%2F03.1-pdf-rag-tool&demo_name=ai-agent&event=VIEW&path=%2F_dbdemos%2Fdata-science%2Fai-agent%2F03-knowledge-base-rag%2F03.1-pdf-rag-tool&version=1">

In [0]:
%pip install -U -qqqq mlflow>=3.1.4 langchain==0.3.27 langgraph==0.6.11 databricks-langchain pydantic databricks-agents unitycatalog-langchain[databricks] databricks-feature-engineering==0.12.1 protobuf<5  cryptography<43 databricks-mcp
dbutils.library.restartPython()

In [0]:
%run ../_resources/01-setup

   

## 1. PDF 정보 추출
Databricks는 AI를 활용하여 PDF 정보를 텍스트로 분석하고 추출하는 내장 `ai_parse_document` 함수를 제공합니다. 이를 통해 비정형 정보를 매우 쉽게 수집할 수 있습니다!

In [0]:
%sql
SELECT path FROM READ_FILES('/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/', format => 'binaryFile') limit 2

In [0]:
%sql
    
-- ai_parse_document는 DBR 17.1 또는 서버리스 런타임에서 사용 가능합니다
SELECT ai_parse_document(content) AS parsed_document
  FROM READ_FILES('/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/', format => 'binaryFile') limit 2

   
## 1.1/ 지식 베이스 테이블 생성

먼저 테이블을 생성하겠습니다. Vector Search를 그 위에 생성할 수 있도록 Change Data Feed를 활성화하겠습니다.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS knowledge_base (
  id BIGINT GENERATED ALWAYS AS IDENTITY,
  product_name STRING,
  title STRING,
  content STRING,
  doc_uri STRING)
  TBLPROPERTIES (delta.enableChangeDataFeed = true);

   

## 1.2/ ai_parse_document를 이용한 PDF 텍스트 변환

이제 Databricks 내장 `ai_parse_document` 함수를 사용하여 PDF 문서를 자동으로 파싱하고 정보를 매우 쉽게 추출하겠습니다!

*참고: 이 경우 비교적 작은 PDF 문서들을 다루므로, RAG 시스템이 제대로 작동하도록 문서의 모든 페이지를 하나의 텍스트 필드로 병합하겠습니다. 더 큰 문서의 경우 컨텍스트 크기를 줄이고 더 많은 문서를 검색/검색할 수 있도록 전처리 단계가 필요할 수 있습니다. 예를 들어, Vector Search를 더 관련성 있게 유지하기 위해 모든 청크에 WIFI 라우터 모델이 포함되도록 하는 등의 작업을 할 수 있습니다.*

In [0]:
%sql
    
INSERT OVERWRITE TABLE knowledge_base (product_name, title, content, doc_uri)
SELECT ai_extract.product_name, ai_extract.title, content, doc_uri
FROM (
  SELECT
    ai_extract(content, array('product_name', 'title')) AS ai_extract,
    content,
    doc_uri
  FROM (
    SELECT array_join(
            transform(parsed_document:document.elements::ARRAY<STRUCT<content:STRING>>, x -> x.content), '\n') AS content,
           path as doc_uri
    FROM (
      SELECT ai_parse_document(content) AS parsed_document, path
      FROM READ_FILES('/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/', format => 'binaryFile') 
      LIMIT 5 -- 데모 비용을 위한 LIMIT 추가 - 실제 작업에서는 제거하세요
    )
  )
);

In [0]:
%sql
SELECT * FROM knowledge_base;

           

## 2/ Vector Search 테이블 생성

### 2.1/ Vector Search 엔드포인트

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/chatbot-rag/rag-basic-prep-2.png?raw=true" style="float: right; margin-left: 10px" width="400px">

Vector Search 엔드포인트는 인덱스가 위치하는 엔티티입니다. 검색 요청을 처리하는 진입점으로 생각하면 됩니다.

첫 번째 Vector Search 엔드포인트를 생성하는 것부터 시작하겠습니다. 생성되면 [Vector Search Endpoints UI](#/setting/clusters/vector-search)에서 확인할 수 있습니다. 엔드포인트 이름을 클릭하면 해당 엔드포인트에서 제공되는 모든 인덱스를 확인할 수 있습니다.

In [0]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

if not endpoint_exists(vsc, VECTOR_SEARCH_ENDPOINT_NAME):
    vsc.create_endpoint(name=VECTOR_SEARCH_ENDPOINT_NAME, endpoint_type="STANDARD")

wait_for_vs_endpoint_to_be_ready(vsc, VECTOR_SEARCH_ENDPOINT_NAME)
print(f"Endpoint named {VECTOR_SEARCH_ENDPOINT_NAME} is ready.")

           

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/chatbot-rag/rag-basic-prep-3.png?raw=true" style="float: right; margin-left: 10px" width="400px">


### 2.2/ Vector Search 인덱스 생성

엔드포인트가 생성되면, 이제 Databricks에 기존 테이블 위에 인덱스를 생성하도록 요청하기만 하면 됩니다.

텍스트 열과 임베딩 파운데이션 모델(`GTE`)을 지정하기만 하면 됩니다. Databricks가 자동으로 인덱스를 구축하고 동기화해 줍니다.

Databricks는 3가지 유형의 Vector Search를 제공합니다:

* **관리형 임베딩**: Databricks가 텍스트 필드로부터 임베딩을 생성하고 Delta 테이블을 인덱스에 동기화합니다 (우리가 사용할 방식)
* **자체 관리 임베딩**: 사용자가 직접 임베딩을 계산하여 Delta 테이블에 저장하고 Databricks가 Delta 테이블을 인덱스에 동기화합니다
* **직접 액세스**: 사용자가 직접 VS 인덱싱을 관리합니다 (Delta 테이블 없음)

이는 API를 사용하거나 Unity Catalog Explorer 메뉴에서 몇 번의 클릭으로 수행할 수 있습니다:

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/index_creation.gif?raw=true" width="600px">

In [0]:
# Free Edtion의 경우, 오류 발생시 UC에서 직접 생성해도 됨
from databricks.sdk import WorkspaceClient

# 인덱싱하려는 테이블
source_table_fullname = f"{catalog}.{dbName}.knowledge_base"
# 인덱스를 저장할 위치
vs_index_fullname = f"{catalog}.{dbName}.knowledge_base_vs_index"

if not index_exists(vsc, VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname):
  print(f"엔드포인트 {VECTOR_SEARCH_ENDPOINT_NAME}에 인덱스 {vs_index_fullname}을 생성하는 중...")
  vsc.create_delta_sync_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=vs_index_fullname,
    source_table_name=source_table_fullname,
    pipeline_type="TRIGGERED",
    primary_key="id",
    embedding_source_column='content', # 텍스트를 포함하는 열
    embedding_model_endpoint_name='databricks-gte-large-en' # 임베딩을 생성하는 데 사용되는 임베딩 엔드포인트
  )
  # 인덱스가 준비되고 모든 임베딩이 생성되고 인덱싱될 때까지 대기합니다
  wait_for_index_to_be_ready(vsc, VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname)
else:
  # 테이블에 저장된 새 데이터로 VS 컨텐츠를 업데이트하기 위해 동기화를 트리거합니다
  wait_for_index_to_be_ready(vsc, VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname)
  vsc.get_index(VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname).sync()

print(f"테이블 {source_table_fullname}의 인덱스 {vs_index_fullname}가 준비되었습니다")

   
## 2.3/ VS 인덱스 테스트: 관련 컨텐츠 검색

이것이 우리가 해야 할 전부입니다. Databricks가 테이블의 새 항목을 자동으로 감지하고 인덱스와 동기화합니다.

데이터셋 크기와 모델 크기에 따라 인덱스 생성이 시작되고 임베딩이 인덱싱되는 데 몇 초가 걸릴 수 있습니다.

유사한 컨텐츠를 검색해 보겠습니다.

*참고: `similarity_search`는 filters 파라미터도 지원합니다. 이는 RAG 시스템에 보안 계층을 추가하는 데 유용합니다. 예를 들어, 호출하는 사람에 따라 일부 민감한 컨텐츠를 필터링할 수 있습니다(예: 사용자 선호도에 따라 특정 부서를 필터링).*

In [0]:
question = "My wifi router gives me error 01, what should I do?"
# question = "와이파이 공유기에서 'error 01' 메시지가 나타납니다. 어떻게 해야 하나요?"
# 현재 embedding 모델은 한국어를 지원하지 않으므로 영어로 질문을 사용합니다. 
# 다국어 지원이 되는 임베딩 모델의 엔드포인트 예는 다음과 같습니다. >> databricks-qwen3-embedding-0-6b 

results = vsc.get_index(VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname).similarity_search(
  query_text=question,
  columns=["id", "content"],
  num_results=1)
docs = results.get('result', {}).get('data_array', [])
docs

   
## 3/ 기존 에이전트에 리트리버를 새 도구로 추가

이제 인덱스가 준비되었으므로, 기존 에이전트에 리트리버를 추가하기만 하면 됩니다!

`agent.py`와 `agent_config.yaml` 파일을 재사용하겠습니다. 리트리버 구성만 추가하면 에이전트가 사용 가능한 도구 중 하나로 추가할 것입니다!

In [0]:
import mlflow
import yaml, sys, os
import mlflow.models
# 현재 작업 디렉터리에 상대적인 ../agent_eval 경로를 추가
agent_eval_path = os.path.abspath(os.path.join(os.getcwd(), "../02-agent-eval"))
sys.path.append(agent_eval_path)
# 모든 트레이스를 한 곳에 보관하기 위해 이전 노트북과 동일한 실험을 사용하겠습니다
mlflow.set_experiment(agent_eval_path+"/02.1_agent_evaluation")
conf_path = os.path.join(agent_eval_path, 'agent_config.yaml')

try:
    config = yaml.safe_load(open(conf_path))
    config["config_version_name"] = "model_with_retriever"
    config["retriever_config"] =  {
        "index_name": vs_index_fullname,
        "tool_name": "product_technical_docs_retriever",
        "num_results": 1,
        "description": "
제품, 인프라, 라우터 및 기타 제품의 기능, 사용법, 문제 해결을 포함한 내부 문서를 검색합니다. 제품 문서 또는 제품 문제에 대한 모든 질문에 사용하십시오."
    }
    yaml.dump(config, open(conf_path, "w"))
except Exception as e:
    print(f"업데이트 건너뜀 - job 실행에서는 무시 - {e}")

In [0]:
%pip install databricks-mcp
# dbutils.library.restartPython()

In [0]:
import sys, os

# Python 재시작 후 경로를 다시 추가
agent_eval_path = os.path.abspath(os.path.join(os.getcwd(), "../02-agent-eval"))
sys.path.append(agent_eval_path)

from agent import AGENT 

# 리트리버를 테스트하여 WIFI 라우터 PDF 가이드에 접근할 수 있는지 확인합니다
request_example = "How do I restart my WIFI router ADSL-R500?"
answer = AGENT.predict({"input":[{"role": "user", "content": request_example}]})

   
이제 `02.1_agent evaluation` 노트북에서와 같이 `mlflow.pyfunc.log_model()`을 사용하여 MLflow 모델 레지스트리에 새 에이전트를 로깅합니다.

In [0]:
# 에이전트 실행에 필요한 리소스를 캡처합니다. 이제 VS 인덱스가 참조되어 있습니다
for r in AGENT.get_resources():
  print(f"Resource: {type(r).__name__}:{r.name}")

In [0]:
with mlflow.start_run(run_name=model_config.get('config_version_name')):
  logged_agent_info = mlflow.pyfunc.log_model(
    name="agent",
    python_model=agent_eval_path+"/agent.py",
    model_config=conf_path,
    input_example={"input": [{"role": "user", "content": request_example}]},
     # 배포를 위한 자동 인증 패스스루를 지정하기 위해 리소스(엔드포인트, 함수, vs 등)를 결정합니다
    resources=AGENT.get_resources(),
    extra_pip_requirements=["databricks-connect"]
    )

   
## 4/ 문서 베이스에 대해 에이전트 평가

새 모델이 준비되었습니다! 평소처럼, 다음 단계는 데이터셋을 평가하여 답변이 개선되고 있는지 확인하는 것입니다.


### 4.1/ 합성 평가 데이터 생성

평가 데이터셋에는 PDF에 대한 항목이 없습니다.

Databricks를 사용하면 합성 평가 데이터로 평가 데이터셋을 쉽게 부트스트랩하고, 시간이 지나면서 이 데이터셋을 개선할 수 있습니다.

In [0]:
from databricks.agents.evals import generate_evals_df

docs = spark.table('knowledge_base')
# 에이전트가 수행하는 작업을 설명합니다
agent_description = """
이 에이전트는 와이파이 라우터, 광케이블 설치, 네트워크 정보와 같은 제품에 대한 기술 질문에 답변하는 RAG 챗봇이며, 고객 유지 전략이나 소셜 미디어 가이드라인에 대한 질문에도 답변합니다. 이 에이전트는 문서 말뭉치에 접근할 수 있으며, 말뭉치에서 관련 문서를 검색하여 유용하고 정확한 답변을 합성하는 것이 주요 역할입니다.
"""

question_guidelines = """
# 사용자 페르소나
- 시스템 문제를 단계별로 해결하는 방법을 묻는 고객
- 내부 정책에 대한 질문을 하는 내부 상담원

# 질문 예시
- 고객이 반품 요청을 제출할 수 없을 때 Error Code 1001: Invalid Return Authorization을 어떻게 해결하나요?
- 설문조사를 배포하려고 할 때 Error Code 1001이 발생합니다. 원인과 해결 방법은 무엇인가요?

# 추가 가이드라인
- 질문은 간결하고 인간적이어야 합니다
"""

# 합성 평가 데이터셋을 생성합니다
evals = generate_evals_df(
    docs,
    # 생성할 평가의 총 개수입니다. 이 메서드는 제공된 문서에 대한 전체 커버리지를 가진 평가를 생성하려고 시도합니다.
    # 이 수가 문서 수보다 적으면,
    # 일부 문서는 평가가 생성되지 않습니다. 자세한 내용은 아래 "num_evals 사용 방법"을 참조하세요.
    num_evals=10
    ,
    # 합성 생성을 안내하는 데 도움이 되는 가이드라인 세트입니다. 이는 생성을 프롬프트하는 데 사용될 자유 형식 문자열입니다.
    agent_description=agent_description,
    question_guidelines=question_guidelines
)
evals["inputs"] = evals["inputs"].apply(lambda x: {"question": x["messages"][0]["content"]})
display(evals)

In [0]:
# 합성 데이터셋을 MLFlow 평가 데이터셋에 추가
eval_dataset_table_name = f"{catalog}.{dbName}.ai_agent_mlflow_eval"

eval_dataset = mlflow.genai.datasets.get_dataset(eval_dataset_table_name)
eval_dataset.merge_records(evals)
print("평가 데이터셋에 레코드가 추가되었습니다.")

### 4.2/ 평가 실행
이전과 마찬가지로 MLFlow 데이터셋을 사용하여 평가를 실행하겠습니다. 모델이 고객 관련 질문에 대해 여전히 제대로 작동하는지, 그리고 이제 지식 베이스 질문에 대해서도 잘 작동하는지 확인하겠습니다!

In [0]:
from mlflow.genai.scorers import RetrievalGroundedness, RelevanceToQuery, Safety, Guidelines
import pandas as pd

eval_dataset = mlflow.genai.datasets.get_dataset(f"{catalog}.{dbName}.ai_agent_mlflow_eval")

# 이전과 동일한 스코어러를 가져옵니다 (함수는 _resources/01-setup에 정의됨, 이전 단계와 유사)
scorers = get_scorers()

# 모델을 로드하고 예측 함수를 생성
loaded_model = mlflow.pyfunc.load_model(f"runs:/{logged_agent_info.run_id}/agent")
def predict_wrapper(question):
    # 챗 스타일 모델에 맞게 포맷팅
    model_input = pd.DataFrame({
        "input": [[{"role": "user", "content": question}]]
    })
    response = loaded_model.predict(model_input)
    return response['output'][-1]['content'][-1]['text']
    
print("평가 실행 중...")
with mlflow.start_run(run_name='eval_with_retriever'):
    results = mlflow.genai.evaluate(data=eval_dataset, predict_fn=predict_wrapper, scorers=scorers)

### 4.3/ 최종 모델 배포! 

준비가 완료되었습니다. 모델을 UC에 배포하고 엔드포인트를 최신 버전으로 업데이트하겠습니다!

In [0]:
from mlflow import MlflowClient
MODEL_NAME = "dbdemos_ai_agent_demo"
UC_MODEL_NAME = f"{catalog}.{dbName}.{MODEL_NAME}"

# 모델을 UC에 등록
client = MlflowClient()
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME, tags={"model": "customer_support_agent", "model_version": "with_retriever"})

client.set_registered_model_alias(name=UC_MODEL_NAME, alias="model-to-deploy", version=uc_registered_model_info.version)
displayHTML(f'<a href="/explore/data/models/{catalog}/{dbName}/{MODEL_NAME}" target="_blank">등록된 에이전트를 보려면 Unity Catalog을 여십시오</a>')

In [0]:
from databricks import agents
# 모델을 리뷰 앱과 모델 서빙 엔드포인트에 배포
endpoint_name = f'{MODEL_NAME}_{catalog}_{db}'[:60]

if len(agents.get_deployments(model_name=UC_MODEL_NAME, model_version=uc_registered_model_info.version)) == 0:
#  agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, endpoint_name=endpoint_name, tags = {"project": "dbdemos"})
  agents.deploy(
    UC_MODEL_NAME, 
    uc_registered_model_info.version, 
    endpoint_name=endpoint_name,  
    scale_to_zero=True, ## Free Edtion의 경우 필수로 활성화
    tags={"project": "dbdemos"}
  )
  print(f"배포 성공: {ENDPOINT_NAME}")

## 다음 단계: Databricks Application에 챗봇 배포

이제 에이전트가 준비되었으므로, GradIO 애플리케이션을 배포하여 최종 사용자에게 컨텐츠를 제공하겠습니다.

[04-deploy-app/04-Deploy-Frontend-Lakehouse-App]($../04-deploy-app/04-Deploy-Frontend-Lakehouse-App) 를 여십시오!